## Extracción de datos climáticos NOAA GHCN-Daily

Este notebook documenta la extracción y preparación de observaciones meteorológicas históricas de NOAA para analizarlas posteriormente con SQL y DuckDB.

**Objetivos**

Obtener datos de una estación meteorológica.
Seleccionar el período comprendido entre el 1 de enero de 2021 y el 31 de diciembre de 2025.
Conservar las temperaturas máximas (TMAX) y mínimas (TMIN).
Convertir las temperaturas a grados Celsius.
Guardar un CSV limpio para las consultas analíticas.
Las consultas SQL se realizarán en un script independiente de Python.

In [1]:
%pip install pandas requests duckdb

Note: you may need to restart the kernel to use updated packages.


## 1. Fuente de datos y extracción

Los datos provienen de **NOAA GHCN-Daily (Global Historical Climatology Network Daily)**, una colección de observaciones meteorológicas diarias mantenida por la National Oceanic and Atmospheric Administration (NOAA).

Fuente oficial: https://www.ncei.noaa.gov/products/land-based-station/global-historical-climatology-network-daily

Estación: USW00094728, ubicada en Central Park, Nueva York.
Variables de interés: temperatura máxima (TMAX) y mínima (TMIN).
Período de análisis: 2021–2025.

El archivo público de la estación contiene registros históricos de muchas fechas y variables. Se descarga y se procesa en memoria; el archivo histórico completo no se guarda en el repositorio.

In [2]:
import requests
import pandas as pd
from io import BytesIO

# Configuración de la extracción
STATION_ID = "USW00094728"
START_DATE = "2021-01-01"
END_DATE = "2025-12-31"

URL = (
    "https://noaa-ghcn-pds.s3.amazonaws.com/"
    f"csv/by_station/{STATION_ID}.csv"
)

# Descargar los datos originales de NOAA
response = requests.get(URL, timeout=120)
response.raise_for_status()

# Leer el CSV y mantener Q_FLAG como texto
raw = pd.read_csv(
    BytesIO(response.content),
    dtype={"Q_FLAG": "string"},
    low_memory=False
)

print("Columnas originales:", raw.columns.tolist())
print("Registros descargados:", len(raw))

display(raw.head())


Columnas originales: ['ID', 'DATE', 'ELEMENT', 'DATA_VALUE', 'M_FLAG', 'Q_FLAG', 'S_FLAG', 'OBS_TIME']
Registros descargados: 458003


,ID,DATE,ELEMENT,DATA_VALUE,M_FLAG,Q_FLAG,S_FLAG,OBS_TIME
0,USW00094728,18690101,TMAX,-17,NaN,<NA>,Z,NaN
1,USW00094728,18690102,TMAX,-28,NaN,<NA>,Z,NaN
2,USW00094728,18690103,TMAX,17,NaN,<NA>,Z,NaN
3,USW00094728,18690104,TMAX,28,NaN,<NA>,Z,NaN
4,USW00094728,18690105,TMAX,61,NaN,<NA>,Z,NaN


## 2. Selección y limpieza

Se seleccionan los registros de la estación indicada dentro del período 2021–2025 y se conservan únicamente `TMAX` y `TMIN`.

Se excluyen las observaciones que tienen una bandera de control de calidad registrada en `Q_FLAG`. NOAA expresa estos valores de temperatura en décimas de grado Celsius, por lo que se divide `DATA_VALUE` por 10.

Después, los registros se reorganizan para obtener una fila por fecha, con columnas separadas para la temperatura máxima y mínima. El resultado conserva las fechas para las que hay datos válidos de ambas variables; no se imputan temperaturas faltantes.


In [3]:
from pathlib import Path

# Seleccionar las columnas necesarias
df = raw[["ID", "DATE", "ELEMENT", "DATA_VALUE", "Q_FLAG"]].copy()

# Convertir DATE a texto para filtrar el período
df["DATE"] = df["DATE"].astype(str)

# Filtrar estación, fechas y temperaturas
df = df[
    (df["ID"] == STATION_ID)
    & (df["DATE"] >= START_DATE.replace("-", ""))
    & (df["DATE"] <= END_DATE.replace("-", ""))
    & (df["ELEMENT"].isin(["TMAX", "TMIN"]))
].copy()

# Conservar observaciones sin bandera de calidad registrada
df = df[df["Q_FLAG"].isna() | (df["Q_FLAG"] == "")].copy()

# Convertir los valores a grados Celsius
df["DATA_VALUE"] = pd.to_numeric(df["DATA_VALUE"], errors="coerce")
df["temperature_c"] = df["DATA_VALUE"] / 10

# Organizar los datos: una fila por fecha con TMAX y TMIN
wide = df.pivot_table(
    index=["ID", "DATE"],
    columns="ELEMENT",
    values="temperature_c",
    aggfunc="first"
).reset_index()

wide.columns.name = None

# Renombrar columnas
wide = wide.rename(columns={
    "ID": "station_id",
    "DATE": "date",
    "TMAX": "tmax_c",
    "TMIN": "tmin_c"
})

# Mantener fechas con ambas temperaturas disponibles
wide = wide.dropna(subset=["tmax_c", "tmin_c"]).copy()

# Convertir la fecha al formato de fecha de Pandas
wide["date"] = pd.to_datetime(wide["date"], format="%Y%m%d")

# Guardar el CSV preparado
Path("data").mkdir(exist_ok=True)
output_path = Path("data/climate_data.csv")
wide.to_csv(output_path, index=False)

print("Archivo generado:", output_path)
print("Cantidad de filas:", len(wide))
print("\nValores nulos por columna:")
print(wide.isna().sum())
print("\nPrimeras filas:")
display(wide.head())


Archivo generado: data\climate_data.csv
Cantidad de filas: 1497

Valores nulos por columna:
station_id    0
date          0
tmax_c        0
tmin_c        0
dtype: int64

Primeras filas:


,station_id,date,tmax_c,tmin_c
0,USW00094728,2021-01-01,4.4,0.6
1,USW00094728,2021-01-02,10.6,2.2
2,USW00094728,2021-01-03,3.3,1.1
3,USW00094728,2021-01-04,6.1,1.7
4,USW00094728,2021-01-05,5.6,2.2


## 3. Resultado de la preparación

El proceso genera data/climate_data.csv con las columnas:

station_id: identificador de la estación meteorológica.

date: fecha de la observación.

tmax_c: temperatura máxima diaria en grados Celsius.

tmin_c: temperatura mínima diaria en grados Celsius.

El archivo obtenido contiene 1.497 fechas con ambas temperaturas disponibles. No representa todos los días calendario del período: las fechas sin datos válidos para una de las dos variables quedan fuera del archivo. No se imputan datos faltantes.

Este CSV será la fuente de las consultas SQL de DuckDB.

## 4. Comprobación de lectura con DuckDB

La siguiente celda comprueba que DuckDB puede consultar directamente el CSV, sin crear una base de datos persistente.

In [4]:
import duckdb

csv_path = "data/climate_data.csv"

# Crear una conexión temporal en memoria
con = duckdb.connect(database=":memory:")

# Leer una muestra del CSV directamente con DuckDB
resultado = con.execute(
    f"SELECT * FROM read_csv_auto('{csv_path}') LIMIT 5"
).df()

display(resultado)

# Comprobar la cantidad de filas que DuckDB detecta en el archivo
total = con.execute(
    f"SELECT COUNT(*) AS total_filas FROM read_csv_auto('{csv_path}')"
).df()

display(total)

con.close()

,station_id,date,tmax_c,tmin_c
0,USW00094728,2021-01-01,4.4,0.6
1,USW00094728,2021-01-02,10.6,2.2
2,USW00094728,2021-01-03,3.3,1.1
3,USW00094728,2021-01-04,6.1,1.7
4,USW00094728,2021-01-05,5.6,2.2


,total_filas
0,1497
